## Understanding Data and Choosing Model

#### Spirals Dataset
- 10000 spiral trajectories
    - Each spiral trajectory has:
        - 100 time points
        - 2 spatial dimensions (x,y)


#### Equation of Dataset (spiral trajectories)
- r=αθ
    - r = radius (distance from the center at a given point in time)
    - **a (alpha, WHAT WE PREDICTING) = spiral constant controlling how tightly the spiral winds and whether it goes clockwise (negative α) or counter-clockwise (positive α)**
    - θ (theta) = angular position (the angle in radians that increases over time as the trajectory moves around the center)


#### Model Options (understanding when to pick which)
1. **ODE-RNN**
    * Combines an ordinary RNN update with a continuous ODE evolution between time points.
    * Best when the data is smooth, regularly sampled, and shows continuous dynamics (like clean spiral motion).
    * Ideal for this dataset if the trajectories are **evenly spaced and have low noise**.
2. **GRU-ODE**
    * Uses a GRU-style gating mechanism inside the ODE to improve stability.
    * Best when sequences are moderately noisy or long, since the gating helps prevent exploding or vanishing gradients.
    * **Suitable if spiral trajectories show small random fluctuations or measurement noise.**
3. **ODE-LSTM**
    * Extends the ODE-RNN with an LSTM’s separate hidden and cell states.
    * Best for complex or irregularly sampled sequences with long-term dependencies.
    * **Useful if the data has missing time steps, variable sampling intervals, or patterns that depend on long-term memory.**

In [1]:
import numpy as np


# Load the dataset
data = np.load("/kaggle/input/spirals/spirals.npz")
xy_train = data["xy_train"]
alpha_train = data["alpha_train"]


# Basic shapes and NaN check
n_samples, seq_len, dims = xy_train.shape
nan_count = np.isnan(xy_train).sum()


# Average step-to-step change (smoothness indicator)
diffs = np.diff(xy_train, axis=1)  # difference between consecutive time steps
step_sizes = np.linalg.norm(diffs, axis=2)  # Euclidean distance between steps
mean_step = step_sizes.mean()
std_step = step_sizes.std()


# Mean curvature / trajectory variability
# curvature ~ angle change rate between successive direction vectors
v1 = diffs[:, :-1, :]
v2 = diffs[:, 1:, :]
dot = np.sum(v1 * v2, axis=2)
norms = np.linalg.norm(v1, axis=2) * np.linalg.norm(v2, axis=2)
cosine_angles = np.clip(dot / (norms + 1e-8), -1.0, 1.0)
angles = np.arccos(cosine_angles)
mean_curvature = np.mean(angles)
std_curvature = np.std(angles)


# Final radius vs α correlation (to confirm dynamics)
radii = np.linalg.norm(xy_train[:, -1, :], axis=1)
corr_radius_alpha = np.corrcoef(radii, alpha_train.flatten())[0, 1]


# Print summary
print("=== Spiral Dataset Summary ===")
print(f"Samples: {n_samples}, Sequence length: {seq_len}, Dimensions: {dims}")
print(f"NaN count: {nan_count}")
print()
print(f"Mean step size: {mean_step:.4f}")
print(f"Std of step size: {std_step:.4f}")
print(f"Mean curvature: {mean_curvature:.4f}")
print(f"Std of curvature: {std_curvature:.4f}")
print(f"Correlation (final radius vs α): {corr_radius_alpha:.4f}")

=== Spiral Dataset Summary ===
Samples: 10000, Sequence length: 100, Dimensions: 2
NaN count: 0

Mean step size: 4.7610
Std of step size: 8.4319
Mean curvature: 1.1846
Std of curvature: 0.7381
Correlation (final radius vs α): 0.0015


## Interpreting Dataset Findings

#### Analyzing Dataset to Choose Optimal Neural ODE Model

I previously discussed how models in the neural ODE family differ in how they handle continuous vs discrete or irregularly spaced data.

**The Dataset Findings**:
* 10000 trajectories, each with 100 time points → ***sequences are long but consistent***.
* 2D spatial data → simple geometry, nothing high-dimensional in the feature space ((x,y) for each time and trajectory).
* NaN count = 0 → ***no missing data***.
* Step size: mean ≈ 4.76, std ≈ 8.43 → ***steps vary, but not extremely chaotic***.
* Curvature: mean ≈ 1.18, std ≈ 0.74 → some variability, but still ***relatively smooth***.
* Correlation final radius vs α ≈ 0.0015 → final radius is basically uncorrelated with α, but that may just mean that α’s influence is uniform over the spiral length (expected for r=αθ).

#### Model choice: **ODE-RNN**
- Simple, efficient, and matches the data's smooth, regularly sampled spirals.
- It will likely train faster than other two models and generalize well.


In [2]:
!pip install equinox
!pip install jax
!pip install optax
!pip install diffrax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.2/193.2 kB 5.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.4/97.4 kB 9.2 MB/s eta 0:00:00


In [4]:
import jax
print("Device:", jax.default_backend())

Device: gpu


In [9]:
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import diffrax

# -------------------------------
# Convert and scale dataset
# -------------------------------
xy_train = jnp.array(xy_train, dtype=jnp.float32)
alpha_train = jnp.array(alpha_train, dtype=jnp.float32)

# Scale XY coordinates to [-1,1]
max_radius = jnp.max(jnp.linalg.norm(xy_train, axis=2))
xy_train = xy_train / max_radius

# Scale alpha to [-1,1]
alpha_max = jnp.max(jnp.abs(alpha_train))
alpha_train = alpha_train / alpha_max

# -------------------------------
# Train/validation split
# -------------------------------
val_frac = 0.1
n_samples = xy_train.shape[0]
n_val = int(n_samples * val_frac)
perm = np.random.permutation(n_samples)

xy_val = xy_train[perm[:n_val]]
alpha_val = alpha_train[perm[:n_val]]
xy_train_split = xy_train[perm[n_val:]]
alpha_train_split = alpha_train[perm[n_val:]]

n_train = xy_train_split.shape[0]

# -------------------------------
# Define ODE function
# -------------------------------
class ODEFunc(eqx.Module):
    W_hh: jnp.ndarray
    W_xh: jnp.ndarray
    b: jnp.ndarray

    def __init__(self, hidden_size, input_size, key):
        k1, k2, k3 = jax.random.split(key, 3)
        self.W_hh = jax.random.normal(k1, (hidden_size, hidden_size), dtype=jnp.float32) * 0.1
        self.W_xh = jax.random.normal(k2, (hidden_size, input_size), dtype=jnp.float32) * 0.1
        self.b = jax.random.normal(k3, (hidden_size,), dtype=jnp.float32) * 0.1

    def __call__(self, t, h, x_t):
        return jnp.tanh(x_t @ self.W_xh.T + h @ self.W_hh.T + self.b)

# -------------------------------
# Define ODE-RNN with learnable h0
# -------------------------------
class ODERNN(eqx.Module):
    hidden_size: int
    input_size: int
    output_size: int
    ode_func: ODEFunc
    linear_out: eqx.nn.Linear
    h0: jnp.ndarray  # learnable initial hidden state

    def __init__(self, hidden_size, input_size, output_size, key):
        k1, k2, k3 = jax.random.split(key, 3)
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.output_size = output_size
        self.ode_func = ODEFunc(hidden_size, input_size, k1)
        self.linear_out = eqx.nn.Linear(hidden_size, output_size, key=k2, dtype=jnp.float32)
        self.h0 = jax.random.normal(k3, (hidden_size,), dtype=jnp.float32) * 0.1

    def forward_single(self, x_seq):
        solver = diffrax.Tsit5()
        term = diffrax.ODETerm(self.ode_func)
        saveat = diffrax.SaveAt(t1=True)

        def step_fn(h, x_t):
            sol = diffrax.diffeqsolve(
                term, solver, t0=0, t1=1.0, dt0=0.1, y0=h, args=x_t, saveat=saveat
            )
            h_new = sol.ys[0]
            h_new = jnp.tanh(h_new + x_t @ self.ode_func.W_xh.T)
            return h_new, None

        h_final, _ = jax.lax.scan(step_fn, self.h0, x_seq)
        return self.linear_out(h_final)

    def __call__(self, x_batch):
        x_batch = x_batch.astype(jnp.float32)
        return jax.vmap(self.forward_single)(x_batch)

# -------------------------------
# Loss function
# -------------------------------
def mse_loss(model, x_batch, y_batch):
    x_batch = x_batch.astype(jnp.float32)
    y_batch = y_batch.astype(jnp.float32)
    preds = model(x_batch)
    return jnp.mean((preds.squeeze() - y_batch) ** 2)

# -------------------------------
# Initialize model and optimizer
# -------------------------------
key = jax.random.PRNGKey(7)
hidden_size = 32 
model = ODERNN(hidden_size, input_size=2, output_size=1, key=key)

optimizer = optax.adam(1e-3)
opt_state = optimizer.init(eqx.filter(model, eqx.is_inexact_array))

# -------------------------------
# Training step
# -------------------------------
trainable_filter = eqx.is_inexact_array

@jax.jit
def update(model, opt_state, x_batch, y_batch):
    grads = jax.grad(lambda m: mse_loss(m, x_batch, y_batch))(
        eqx.filter(model, trainable_filter)
    )
    updates, opt_state = optimizer.update(grads, opt_state)
    model = eqx.apply_updates(model, updates)
    return model, opt_state

# -------------------------------
# Training loop with early stopping
# -------------------------------
batch_size = 32
max_steps = 5000
patience = 50
best_val_loss = float("inf")
steps_without_improve = 0

for step in range(max_steps):
    idx = np.random.choice(n_train, batch_size, replace=False)
    x_batch = jnp.array(xy_train_split[idx], dtype=jnp.float32)
    y_batch = jnp.array(alpha_train_split[idx], dtype=jnp.float32)
    model, opt_state = update(model, opt_state, x_batch, y_batch)

    if step % 10 == 0:
        val_loss = mse_loss(model, xy_val, alpha_val)
        print(f"Step {step}, Validation Loss: {val_loss:.6f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            steps_without_improve = 0
            best_model = eqx.tree_at(lambda m: m, model, replace=model)
        else:
            steps_without_improve += 10

    if steps_without_improve >= patience:
        print(f"Early stopping at step {step}")
        model = best_model
        break

# -------------------------------
# After training, rescale alpha predictions
# -------------------------------
alpha_pred_scaled = model(xy_val) * alpha_max
print("Predicted alpha shape:", alpha_pred_scaled.shape)

Step 0, Validation Loss: 0.023523
Step 10, Validation Loss: 0.005126
Step 20, Validation Loss: 0.004965
Step 30, Validation Loss: 0.004661
Step 40, Validation Loss: 0.004506
Step 50, Validation Loss: 0.004457
Step 60, Validation Loss: 0.004453
Step 70, Validation Loss: 0.004450
Step 80, Validation Loss: 0.004494
Step 90, Validation Loss: 0.004449
Step 100, Validation Loss: 0.004555
Step 110, Validation Loss: 0.004480
Step 120, Validation Loss: 0.004460
Step 130, Validation Loss: 0.004451
Step 140, Validation Loss: 0.004680
Early stopping at step 140
Predicted alpha shape: (1000, 1)


## Model Decisions Made

#### Params chosen for the model:
- **Hidden Size (number of neurons per layer) = 32**
    - Beyond 128 is high risk of overfitting. 32 is medium between simple and complex dynamic representation.
- **Number of Layers (how many stacked ODE-RNN cells) = 1**
    - Data is simple so 1 layer is enough (more efficient)
- **Activation Function (non linear function at the end of layers)**
    - Tanh for both ODE updates and RNN updates
    - Tanh is standard for both because it keeps hidden states stable (not exploding or vanishing) and bounded (output always between -1 and 1). Other options were ReLU, GELU, ELU
- **Batch size (number of training samples processed together in single forward/backward pass in NN) = 32**
    - Larger batch sizes can stabilize training but use more memory. For 10,000 samples, batch sizes 64–256 are reasonable. 
    - Smaller batch size = noisier gradient updates, which can sometimes help generalization.
- **Epochs = 32**
    - Largely dependent on batch size. With smooth, deterministic spirals, it may reach good performance even at 30–50 epochs. 
    - I am using a validation split and early stopping to avoid overfitting.
- **ODE solver = Tsit5**
    - Accurate and fast for smooth dynamics
- **Initial hidden state of the ODE-RNN = initialized to be learnable **
    - Initial y0 is typically an array of zeros for this type of model (ODE-RNN), but here it s learnable to potentially improve performance.
- **Learning rate = 1e-3 with Adam optimizer**
    - Adam works well for this type of regression
    - 1e-3 is the standard starting point
- **RNN update rule: h = tanh(h + x_t @ W_xh.T)**
    - H = memory of trajectory up to this point in time (final h is new memory and h inside equation is the old memory)
    - x_t = new x,y coordinates being learned (input)
    - W_xh = weights mapping input to hidden state (memory)
    - ***Summary: “New memory” = “old memory” + “new information scaled by the memory weights”***

#### Fun Facts about Model:
- ***10% of the dataset was set aside as a validation (test) set*** to monitor performance and prevent overfitting.
- The ***@jax.jit compiler*** optimizes and compiles the training step into efficient, fused machine code, making repeated updates much faster on CPU or GPU. (Recommendation of professor last lesson)
- I ***used GPU T4x2 on Kaggle to run*** this model. This likely speeded up operations compared to the CPU on Jupiter Notebook.
- The model uses only the final hidden state after processing all timesteps to predict alpha, not the entire hidden trajectory. Because the final hidden state contains a summarized representation of all previous inputs, making it sufficient for predicting alpha without needing the full sequence of intermediate states.

In [14]:
# -------------------------------
# Predict alpha for the test set
# -------------------------------

# Load test data from original npz
xy_test = jnp.array(data["xy_test"], dtype=jnp.float32)

# If you scaled training by max_radius, scale test the same way
xy_test_scaled = xy_test / max_radius  # make sure max_radius is defined from training set

# Predict (no extra vmap if model already batch-compatible)
alpha_pred_test = model(xy_test_scaled)  # shape (n_test, 1)

# If you scaled alpha during training, rescale predictions
alpha_pred_test = alpha_pred_test * alpha_max  # make sure alpha_max is defined

# Convert to numpy array and save
alpha_pred_test = np.array(alpha_pred_test)
np.save("alpha_pred_test.npy", alpha_pred_test)

# Confirm shape
print("Saved predictions shape:", alpha_pred_test.shape)

Saved predictions shape: (10000, 1)
